# Planner

> An MPC planner with communication.

In [ ]:
#| default_exp planners.mpc

In [ ]:
#| export
import torch
import torch.nn as nn
import torch.nn.functional as F

## JEPA Planner

In [ ]:
#| export
class JEPAGoalPlanner:
    def __init__(self, model, action_dim= 4, horizon=3, history_size=3,
                 pop_size=300, topk=30, opt_steps=30, device='cpu'):
        self.model = model
        self.action_dim = action_dim
        self.horizon = horizon
        self.history_size = history_size
        self.pop_size = pop_size
        self.topk = topk
        self.opt_steps = opt_steps
        self.device = device

    def _sample_actions(self, probs):
        # probs: (B, horizon, action_dim) -> samples: (B, S, horizon, action_dim) one-hot
        B, H, A = probs.shape
        flat = probs.unsqueeze(1).expand(B, self.pop_size, H, A).reshape(-1, A)
        idx = torch.multinomial(flat, 1).view(B, self.pop_size, H)
        return idx.float()
        #return F.one_hot(idx, num_classes=A).float()

    def _update_dist(self, cost, samples):
        # cost: (B, S), samples: (B, S, horizon) -- raw action indices (dimensionless)
        B, S, H = samples.shape
        A = self.action_dim  # must come from self, since samples no longer carries it
        new_probs = torch.zeros(B, H, A, device=cost.device)
        for b in range(B):
            elite_idx = torch.topk(-cost[b], self.topk).indices
            elites = samples[b, elite_idx]  # (topk, H) -- raw indices
            for t in range(H):
                counts = torch.bincount(elites[:, t].long(), minlength=A).float()
                new_probs[b, t] = (counts + 1e-6) / (counts.sum() + 1e-6 * A)
        return new_probs

    @torch.no_grad()
    def plan(self, info_dict):
        """
        info_dict must contain (per JEPA.get_cost / rollout requirements):
          - pixels: (B, history_size, C, H, W)   recent observation history
          - action: (B, history_size, action_dim) actions taken during that history
          - goal:   (B, C, H, W)                  goal observation
          - msg_indices (optional): (B, 1, 49)    message valid only for the first
                                                    predicted step (per model.rollout)
        Returns the first action of the best plan per batch element, plus the full plan.
        """
        device = self.device
        B = info_dict["pixels"].size(0)
        probs = torch.full((B, self.horizon, self.action_dim), 1.0 / self.action_dim, device=device)
        print("Starting planning with {} optimization steps...".format(self.opt_steps))
        for _ in range(self.opt_steps):
            samples = self._sample_actions(probs)  # (B, S, horizon, action_dim)

            # full action sequence fed to rollout = known history actions + sampled future actions
            hist_action = info_dict["action"].unsqueeze(1).expand(
                B, self.pop_size, *info_dict["action"].shape[1:]
            )
            action_candidates = torch.cat([hist_action, samples], dim=2).long().to(device)

            # expand the rest of info_dict over the sample (S) dimension
            cand_info = {
                k: (v.unsqueeze(1).expand(B, self.pop_size, *v.shape[1:]).to(device)
                    if torch.is_tensor(v) else v)
                for k, v in info_dict.items()
            }
            
            cost = self.model.get_cost(cand_info, action_candidates)  # (B, S)
            probs = self._update_dist(cost, samples)
            print("Optimization step completed.")
        
        plan = torch.argmax(probs, dim=-1)       # (B, horizon)
        first_action = plan[:, 0]
        return first_action, plan
    
    @torch.no_grad()
    def eval_plan(self, info_dict, device, plan):
        
        final_action_seq = torch.cat(
        [info_dict["action"],plan], #F.one_hot(plan, num_classes=self.action_dim).float()],
        dim=1,
        ).unsqueeze(1).to(device)  # (B, 1, H+horizon, action_dim)

        final_info = {
            k: (v.unsqueeze(1).to(device) if torch.is_tensor(v) else v)
            for k, v in info_dict.items()
        }
        final_cost = self.model.get_cost(final_info, final_action_seq).squeeze(1)  # (B,)

        # first_action = plan[:, 0]
        return final_cost
        

### Improved CEM Solver

In [ ]:
#| export
import torch
import torch.nn.functional as F


class DiscreteCEMPlanner:
    """
    CEM-based planner for a JEPA-style dynamics model whose action encoder
    consumes raw integer indices (nn.Embedding), not one-hot vectors.

    Improvements over the original implementation:
      - Gumbel-max sampling instead of torch.multinomial (matches
        CategoricalCEMSolver, vectorized, uses a seeded generator).
      - Elitism anchor: the first sample each step is forced to the current
        argmax action sequence ("mean candidate"), so the best-known plan is
        always re-evaluated.
      - Fully vectorized elite refit: no Python loops over batch or horizon.
        One-hot is used ONLY over the small elite set (topk, not pop_size)
        purely to compute frequency counts -- it is never passed to the model.
      - Optional Laplace smoothing and EMA momentum on the probability update.
      - info_dict is expanded across the sample dimension once, outside the
        optimization loop, instead of being rebuilt every step.
      - Candidates fed to model.get_cost remain raw long indices throughout,
        matching what DiscreteActionEncoder expects.
    """

    def __init__(self, model, action_dim=4, horizon=3, history_size=3,
                 pop_size=300, topk=30, opt_steps=30,
                 smoothing=0.0, alpha=0.0, device='cpu', seed=1234):
        self.model = model
        self.action_dim = action_dim
        self.horizon = horizon
        self.history_size = history_size
        self.pop_size = pop_size
        self.topk = topk
        self.opt_steps = opt_steps
        self.smoothing = smoothing
        self.alpha = alpha
        self.device = device
        self.torch_gen = torch.Generator(device=device).manual_seed(seed)

    def _sample_actions(self, probs):
        """
        Gumbel-max sample of categorical action indices, with the first
        sample forced to the current distribution's argmax (elitism anchor).

        probs: (B, horizon, action_dim)
        returns: indices (B, pop_size, horizon), long
        """
        B, H, A = probs.shape
        log_probs = probs.clamp_min(1e-10).log()
        log_probs = log_probs.unsqueeze(1).expand(B, self.pop_size, H, A)

        u = torch.rand(
            log_probs.shape, generator=self.torch_gen,
            device=self.device, dtype=probs.dtype,
        ).clamp_min(1e-10)
        gumbel = -(-u.log()).log()

        indices = (log_probs + gumbel).argmax(dim=-1)  # (B, S, H)
        indices[:, 0] = probs.argmax(dim=-1)            # elitism anchor
        return indices

    def _update_dist(self, cost, samples):
        """
        Vectorized elite refit (no Python loops over batch or horizon).

        cost: (B, S) -- lower is better
        samples: (B, S, H) -- raw action indices, long
        returns: new_probs (B, H, action_dim)
        """
        B, S, H = samples.shape
        A = self.action_dim

        # elites: lowest-cost candidates
        _, topk_inds = torch.topk(cost, k=self.topk, dim=1, largest=False)
        batch_idx = torch.arange(B, device=cost.device).unsqueeze(1).expand(-1, self.topk)
        elites = samples[batch_idx, topk_inds]  # (B, topk, H)

        # one-hot ONLY over the small elite set, purely for frequency counting
        # (this is never fed to the model -- get_cost always sees raw indices)
        elite_onehot = F.one_hot(elites, num_classes=A).float()  # (B, topk, H, A)
        new_probs = elite_onehot.mean(dim=1)  # (B, H, A)

        if self.smoothing > 0:
            new_probs = new_probs + self.smoothing
            new_probs = new_probs / new_probs.sum(dim=-1, keepdim=True)

        return new_probs

    @torch.no_grad()
    def plan(self, info_dict, verbose=False):
        """
        info_dict must contain (per JEPA.get_cost / rollout requirements):
          - pixels: (B, history_size, C, H, W)   recent observation history
          - action: (B, history_size)            raw action indices for history
          - goal:   (B, C, H, W)                  goal observation
          - msg_indices (optional): (B, 1, 49)

        Returns the first action of the best plan per batch element, plus the
        full plan.
        """
        device = self.device
        B = info_dict["pixels"].size(0)
        probs = torch.full(
            (B, self.horizon, self.action_dim), 1.0 / self.action_dim, device=device
        )

        # Expand info_dict (and history actions) across the sample dimension
        # ONCE -- these don't change across optimization steps, unlike the
        # sampled future actions.
        cand_info = {
            k: (v.unsqueeze(1).expand(B, self.pop_size, *v.shape[1:]).to(device)
                if torch.is_tensor(v) else v)
            for k, v in info_dict.items()
        }
        hist_action = info_dict["action"].unsqueeze(1).expand(
            B, self.pop_size, *info_dict["action"].shape[1:]
        ).to(device)

        for step in range(self.opt_steps):
            samples = self._sample_actions(probs)  # (B, S, horizon), long

            # known history + sampled future actions, still raw indices
            action_candidates = torch.cat([hist_action, samples], dim=2).long()

            cost = self.model.get_cost(cand_info, action_candidates)  # (B, S)
            new_probs = self._update_dist(cost, samples)

            probs = (
                self.alpha * probs + (1 - self.alpha) * new_probs
                if self.alpha > 0 else new_probs
            )

            if verbose:
                print(
                    f"[DiscreteCEMPlanner] step {step + 1}/{self.opt_steps} "
                    f"mean elite cost: {cost.mean().item():.4f}"
                )

        plan = torch.argmax(probs, dim=-1)  # (B, horizon)
        first_action = plan[:, 0]
        return first_action, plan

    @torch.no_grad()
    def eval_plan(self, info_dict, device, plan):
        final_action_seq = torch.cat(
            [info_dict["action"], plan], dim=1
        ).unsqueeze(1).to(device).long()  # (B, 1, H+horizon)

        final_info = {
            k: (v.unsqueeze(1).to(device) if torch.is_tensor(v) else v)
            for k, v in info_dict.items()
        }
        final_cost = self.model.get_cost(final_info, final_action_seq).squeeze(1)  # (B,)
        return final_cost

### Categorical CEM Solver

In [ ]:
import time
from typing import Any

import gymnasium as gym
import numpy as np
import torch

from .callbacks import Callback
from .solver import Costable


class CategoricalCEMSolver:
    """Cross Entropy Method solver for discrete action optimization.

    Maintains a per-timestep categorical distribution over discrete actions,
    samples candidate trajectories via Gumbel-max, and refits the distribution
    from the top-K elites' empirical frequencies.

    Args:
        cost: Cost object to plan against (a Costable, e.g. a ShootingCostEvaluator).
        batch_size: Number of environments to process in parallel.
        num_samples: Number of action candidates to sample per iteration.
        n_steps: Number of CEM iterations.
        topk: Number of elite samples to keep for distribution update.
        smoothing: Laplace smoothing added to refit probs to avoid collapse.
        alpha: Momentum for probs EMA update (0 = full overwrite).
        device: Device for tensor computations.
        seed: Random seed for reproducibility.
        callbacks: Optional list of callbacks.
    """

    def __init__(
        self,
        cost: Costable,
        batch_size: int = 1,
        num_samples: int = 300,
        n_steps: int = 30,
        topk: int = 30,
        smoothing: float = 0.0,
        alpha: float = 0.0,
        device: str | torch.device = 'cpu',
        seed: int = 1234,
        callbacks: list[Callback] | None = None,
    ) -> None:
        self.cost = cost
        self.batch_size = batch_size
        self.num_samples = num_samples
        self.n_steps = n_steps
        self.topk = topk
        self.smoothing = smoothing
        self.alpha = alpha
        self.device = device
        self.torch_gen = torch.Generator(device=device).manual_seed(seed)
        self.callbacks = list(callbacks) if callbacks else []
        try:
            self._dtype = next(cost.parameters()).dtype
        except (AttributeError, StopIteration):
            self._dtype = torch.float32

    def configure(
        self, *, action_space: gym.Space, n_envs: int, config: Any
    ) -> None:
        """Configure the solver with environment specifications."""

        assert isinstance(
            action_space, gym.spaces.MultiDiscrete
        ) or isinstance(action_space, gym.spaces.Discrete), (
            f'Action space must be MultiDiscrete or Discrete, got {type(action_space)}'
        )
        self._action_space = action_space
        self._n_envs = n_envs
        self._config = config
        self._base_simplex_dim = (
            int(action_space.nvec[0])
            if isinstance(action_space, gym.spaces.MultiDiscrete)
            else int(action_space.n)
        )
        self._configured = True

    @property
    def n_envs(self) -> int:
        return self._n_envs

    @property
    def action_dim(self) -> int:
        return self._base_simplex_dim

    @property
    def action_block(self) -> int:
        return self._config.action_block

    @property
    def base_simplex_dim(self) -> int:
        """Number of categories per action position."""
        return self._base_simplex_dim

    @property
    def action_simplex_dim(self) -> int:
        """Flattened simplex dim including action_block grouping."""
        return self._base_simplex_dim * self.action_block

    @property
    def horizon(self) -> int:
        return self._config.horizon

    @property
    def dtype(self) -> torch.dtype:
        return self._dtype

    def __call__(self, *args: Any, **kwargs: Any) -> dict:
        return self.solve(*args, **kwargs)

    def init_probs(self, n_envs: int) -> torch.Tensor:
        """Initialize uniform categorical probabilities.

        Shape: (n_envs, horizon, action_block, base_simplex_dim).
        """
        K = self._base_simplex_dim
        return torch.full(
            (n_envs, self.horizon, self.action_block, K),
            1.0 / K,
            dtype=self.dtype,
            device=self.device,
        )

    def _sample_indices(self, probs: torch.Tensor) -> torch.Tensor:
        """Gumbel-max sample of categorical indices.

        Args:
            probs: shape (B, H, action_block, K).

        Returns:
            indices: shape (B, num_samples, H, action_block).
        """
        bs, H, ab, K = probs.shape
        log_probs = probs.clamp_min(1e-10).log()
        log_probs = log_probs.unsqueeze(1).expand(
            bs, self.num_samples, H, ab, K
        )
        u = torch.rand(
            log_probs.shape,
            generator=self.torch_gen,
            device=self.device,
            dtype=self.dtype,
        ).clamp_min(1e-10)
        gumbel = -(-u.log()).log()
        return (log_probs + gumbel).argmax(dim=-1)

    @torch.inference_mode()
    def solve(self, info_dict: dict, init_action: Any = None) -> dict:
        """Solve the planning problem using Categorical CEM.

        ``init_action`` is accepted for API parity with other solvers but is
        ignored; probs are always initialized uniform.
        """
        del init_action
        start_time = time.time()
        outputs: dict = {'costs': [], 'probs': []}

        total_envs = len(next(iter(info_dict.values())))
        probs = self.init_probs(total_envs)

        for cb in self.callbacks:
            cb.reset()

        for start_idx in range(0, total_envs, self.batch_size):
            end_idx = min(start_idx + self.batch_size, total_envs)
            current_bs = end_idx - start_idx
            batch_probs = probs[start_idx:end_idx]

            expanded_infos: dict = {}
            for k, v in info_dict.items():
                v_batch = v[start_idx:end_idx]
                if torch.is_tensor(v):
                    target_dtype = (
                        self.dtype if v_batch.is_floating_point() else None
                    )
                    v_batch = (
                        v_batch.to(device=self.device, dtype=target_dtype)
                        .unsqueeze(1)
                        .expand(
                            current_bs,
                            self.num_samples,
                            *v_batch.shape[1:],
                        )
                    )
                elif isinstance(v, np.ndarray):
                    v_batch = np.repeat(
                        v_batch[:, None, ...], self.num_samples, axis=1
                    )
                expanded_infos[k] = v_batch

            for cb in self.callbacks:
                cb.start_batch()

            final_batch_cost = None
            for step in range(self.n_steps):
                # Sample indices: (B, N, H, action_block)
                indices = self._sample_indices(batch_probs)

                # Force first sample to argmax of current probs (analog of CEM's "current mean")
                indices[:, 0] = batch_probs.argmax(dim=-1)

                # One-hot for cost: (B, N, H, action_block, K) -> flatten last two
                one_hot = torch.nn.functional.one_hot(
                    indices, num_classes=self._base_simplex_dim
                ).to(self.dtype)
                candidates = one_hot.reshape(
                    current_bs,
                    self.num_samples,
                    self.horizon,
                    self.action_simplex_dim,
                )

                costs = self.cost.get_cost(expanded_infos, candidates)

                assert isinstance(costs, torch.Tensor), (
                    f'Expected cost to be a torch.Tensor, got {type(costs)}'
                )
                assert (
                    costs.ndim == 2
                    and costs.shape[0] == current_bs
                    and costs.shape[1] == self.num_samples
                ), (
                    f'Expected cost to be of shape ({current_bs}, {self.num_samples}), got {costs.shape}'
                )

                topk_vals, topk_inds = torch.topk(
                    costs, k=self.topk, dim=1, largest=False
                )

                batch_indices = (
                    torch.arange(current_bs, device=self.device)
                    .unsqueeze(1)
                    .expand(-1, self.topk)
                )
                # (B, K_topk, H, action_block, K_simplex)
                topk_one_hot = one_hot[batch_indices, topk_inds]

                # Refit: empirical frequencies over the elite set
                new_probs = topk_one_hot.mean(dim=1)

                if self.smoothing > 0:
                    new_probs = new_probs + self.smoothing
                    new_probs = new_probs / new_probs.sum(dim=-1, keepdim=True)

                prev_probs = batch_probs
                if self.alpha > 0:
                    batch_probs = (
                        self.alpha * batch_probs + (1 - self.alpha) * new_probs
                    )
                else:
                    batch_probs = new_probs

                for cb in self.callbacks:
                    cb(
                        step=step,
                        candidates=candidates,
                        costs=costs,
                        topk_vals=topk_vals,
                        topk_inds=topk_inds,
                        topk_candidates=topk_one_hot,
                        probs=batch_probs,
                        prev_probs=prev_probs,
                    )

                final_batch_cost = topk_vals.mean(dim=1).cpu().tolist()

            probs[start_idx:end_idx] = batch_probs
            outputs['costs'].extend(final_batch_cost)

        # Output discrete actions: argmax of final probs.
        # Shape (n_envs, horizon, action_block) — matches PGDSolver convention.
        actions = probs.argmax(dim=-1)

        outputs['actions'] = actions.detach().cpu()
        outputs['probs'] = [probs.detach().cpu()]

        if self.callbacks:
            outputs['callbacks'] = {}
            for cb in self.callbacks:
                cb.end_solve()
                outputs['callbacks'][cb.output_key] = cb.history

        print(
            f'Categorical CEM solve time: {time.time() - start_time:.4f} seconds'
        )
        return outputs